### Cell 1: Install packages

**What runs:** installs the required packages into the kernel's Python. `scikit-learn<1.8` is needed because newer versions produce different `StratifiedGroupKFold` folds, so results would not match the original run. `fair-esm` runs ESM-2. If `import torch` fails afterwards, install PyTorch first (CUDA build for an NVIDIA GPU, from pytorch.org, or the CPU build). After this cell, restart the kernel and continue from Cell 2.

In [ ]:
%pip install -q "scikit-learn<1.8" biopython fair-esm xgboost matplotlib scipy pandas joblib
import sklearn
print("scikit-learn:", sklearn.__version__)
if tuple(map(int, sklearn.__version__.split(".")[:2])) >= (1, 8):
    print("scikit-learn is still 1.8+. Restart the kernel and run this cell again once.")
try:
    import torch
    print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
except Exception as e:
    print("torch not found, install PyTorch first:", e)
print("Now: Kernel -> Restart, then continue from Cell 2 (skip Cell 1).")

### Cell 2: Imports and constants


In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")   # avoids the OpenMP duplicate-library error on Windows/Anaconda
import os, sys, time
# ---- notebook cell 6 ----
import argparse
import json
import math
import pickle
import warnings
import hashlib
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from Bio.Align import substitution_matrices
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from Bio.SeqUtils.ProtParamData import kd
from scipy.stats import loguniform, randint, t
from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin, clone
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    confusion_matrix, f1_score, matthews_corrcoef, precision_recall_curve, roc_auc_score,
)
from sklearn.model_selection import (
    GroupKFold, RandomizedSearchCV, RepeatedStratifiedKFold,
    StratifiedGroupKFold, StratifiedKFold, cross_val_predict,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, NuSVC

import sklearn
_SKLEARN_VERSION = tuple(map(int, sklearn.__version__.split('.')[:2]))
_HISTGB_HAS_CLASS_WEIGHT = _SKLEARN_VERSION >= (1, 2)

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    XGBClassifier = None
    HAS_XGB = False

try:
    import torch
    import esm
    HAS_ESM = True
except Exception:
    HAS_ESM = False

try:
    import shap
    HAS_SHAP = True
except Exception:
    HAS_SHAP = False

try:
    import lime
    import lime.lime_tabular
    HAS_LIME = True
except Exception:
    HAS_LIME = False

STANDARD_AA = "ACDEFGHIKLMNPQRSTVWY"
AA_SET = set(STANDARD_AA)
ALL_DIPEPTIDES = [a + b for a in STANDARD_AA for b in STANDARD_AA]
BLOSUM62 = substitution_matrices.load("BLOSUM62")

ESM_MODEL_INFO = {
    "esm2_t6_8M_UR50D": 6,
    "esm2_t12_35M_UR50D": 12,
    "esm2_t30_150M_UR50D": 30,
    "esm2_t33_650M_UR50D": 33,
}

print(f"sklearn: {sklearn.__version__} | XGB: {HAS_XGB} | ESM: {HAS_ESM} | SHAP: {HAS_SHAP} | LIME: {HAS_LIME}")

### Cell 3: Config
**What runs:** defines the `Config` class with all settings (similarity threshold, ESM model, CV folds, hyper-parameter search) and the CTD/physicochemical tables. Definitions only.

In [ ]:
# ---- notebook cell 8 ----
@dataclass
class Config:
    input_csv: str = "main_dataset_cleaned_2.csv"
    outdir: str = "ml_report_v5"
    seq_col: str = "Peptide"
    label_col: str = "label"
    id_col: str = "dataset_id"

    # MHC restriction
    mhc_rank_col: str = "MHC_Percentile_Rank"
    mhc_ic50_col: str = "MHC_IC50_nM"
    mhc_class_col: str = "MHC_Class_Used"
    mhc_allele_col: str = "MHC_Allele_Best"
    require_mhc_assay: bool = True
    require_known_class: bool = True
    require_tcell_assay: bool = True
    assay_col: str = "Assay"
    tcell_assay_keyword: str = "T cell assay"

    # Similarity / conflict
    similarity_threshold: float = 0.80
    conflict_consensus_fraction: float = 0.75
    resolve_conflicts: bool = True

    # Features
    include_dpc: bool = True          # CHANGED: was False. Tested: small but free (+0.002-0.01 AUC)
    include_anchor_features: bool = True   # NEW: MHC-class-aware anchor-position features
    autoc_maxlag: int = 5
    use_mhc_features: bool = True
    mhc_allele_min_count: int = 5

    # ESM-2
    use_esm_embeddings: bool = True
    esm_model_name: str = "esm2_t12_35M_UR50D"
    esm_repr_layer: int = -1
    esm_batch_size: int = 16
    esm_device: str = ""

    # Selection
    var_threshold: float = 1e-5
    corr_threshold: float = 0.90
    mi_top_k: int = 300               # CHANGED: was 250. Modest headroom increase.

    # CV / HPO
    outer_folds: int = 5
    outer_repeats: int = 3
    inner_folds: int = 3
    use_similarity_cv: str = "auto"
    random_search_iter: int = 35      # CHANGED: was 25.
    threshold_metric: str = "mcc"
    ensemble_min_inner_auc: float = 0.70

    # NEW: explainability / publication artifacts
    save_explainability_artifacts: bool = True
    shap_max_background: int = 250
    lime_n_explanations: int = 3

    random_seed: int = 42
    n_jobs: int = 4

    def __post_init__(self):
        if self.esm_repr_layer is None or self.esm_repr_layer < 0:
            self.esm_repr_layer = ESM_MODEL_INFO.get(self.esm_model_name, 12)


CTD_GROUPS = {
    "Hydrophobicity": {1: set("RKEDQN"), 2: set("GASTPHY"), 3: set("CLVIMFW")},
    "NormVanDerWaalsVolume": {1: set("GASTPDC"), 2: set("NVEQIL"), 3: set("MHKFRYW")},
    "Polarity": {1: set("LIFWCMVY"), 2: set("PATGS"), 3: set("HQRKNED")},
    "Polarizability": {1: set("GASDT"), 2: set("CPNVEQIL"), 3: set("KMHFRYW")},
    "Charge": {1: set("KR"), 2: set("ANCQGHILMFPSTWYV"), 3: set("DE")},
    "SecondaryStructure": {1: set("EALMQKRH"), 2: set("VIYCWFT"), 3: set("GNPSD")},
    "SolventAccessibility": {1: set("ALFCGIVW"), 2: set("RKQEND"), 3: set("MSPTHY")},
}

AA_POLARITY_GRANTHAM = {
    "A": 8.1, "R": 10.5, "N": 11.6, "D": 13.0, "C": 5.5, "Q": 10.5, "E": 12.3, "G": 9.0,
    "H": 10.4, "I": 5.2, "L": 4.9, "K": 11.3, "M": 5.7, "F": 5.2, "P": 8.0, "S": 9.2,
    "T": 8.6, "W": 5.4, "Y": 6.2, "V": 5.9,
}

AA_VOLUME_ZAMYATNIN = {
    "A": 88.6, "R": 173.4, "N": 114.1, "D": 111.1, "C": 108.5, "Q": 143.8, "E": 138.4,
    "G": 60.1, "H": 153.2, "I": 166.7, "L": 166.7, "K": 168.6, "M": 162.9, "F": 189.9,
    "P": 112.7, "S": 89.0, "T": 116.1, "W": 227.8, "Y": 193.6, "V": 140.0,
}

HYDROPHOBIC_AA = set("AILMFWVC")

### Cell 4: Sequence cleaning and similarity groups



In [ ]:
# ---- notebook cell 10 ----
def clean_sequence(seq: str) -> str:
    return str(seq).strip().upper()


def is_valid_peptide(seq: str) -> bool:
    seq = clean_sequence(seq)
    return len(seq) > 0 and set(seq).issubset(AA_SET)


class UnionFind:
    def __init__(self, n: int):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x: int) -> int:
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a: int, b: int):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1


def normalized_edit_identity(a: str, b: str, threshold_hint: float = 0.0) -> float:
    a, b = clean_sequence(a), clean_sequence(b)
    if a == b:
        return 1.0
    la, lb = len(a), len(b)
    max_len = max(la, lb)
    if max_len == 0:
        return 0.0
    if min(la, lb) / max_len < threshold_hint:
        return min(la, lb) / max_len
    prev = list(range(lb + 1))
    for i, ca in enumerate(a, start=1):
        curr = [i]
        for j, cb in enumerate(b, start=1):
            curr.append(min(prev[j] + 1, curr[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = curr
    return 1.0 - prev[-1] / max_len


def compute_similarity_groups(seqs: list, threshold: float) -> np.ndarray:
    """Union-find clusters of near-duplicate sequences (identity >= threshold)."""
    n = len(seqs)
    if n > 3000:
        print(f"[WARNING] O(n^2) similarity clustering on {n} sequences will be very slow.")
        print("          Consider pre-clustering with cd-hit or mmseqs for larger datasets.")

    uf = UnionFind(n)
    for i in range(n):
        if i % 250 == 0 and i > 0:
            print(f"[similarity] compared {i}/{n} sequences")
        for j in range(i + 1, n):
            if min(len(seqs[i]), len(seqs[j])) / max(len(seqs[i]), len(seqs[j])) < threshold:
                continue
            if normalized_edit_identity(seqs[i], seqs[j], threshold_hint=threshold) >= threshold:
                uf.union(i, j)
    roots = [uf.find(i) for i in range(n)]
    root_to_id = {root: idx for idx, root in enumerate(sorted(set(roots)))}
    return np.array([root_to_id[root] for root in roots])

### Cell 5: Classical features, part 1

**What runs:** functions that compute AAC, DPC, CTD and physicochemical features. Definitions only.

In [ ]:
# ---- notebook cell 12 ----
def aac_features(seq: str) -> dict:
    seq = clean_sequence(seq)
    counts = Counter(seq)
    length = len(seq)
    return {f"AAC_{aa}": counts.get(aa, 0) / length * 100 for aa in STANDARD_AA}


def dpc_features(seq: str) -> dict:
    seq = clean_sequence(seq)
    denom = max(len(seq) - 1, 1)
    counts = Counter(seq[i:i + 2] for i in range(len(seq) - 1))
    return {f"DPC_{dp}": counts.get(dp, 0) / denom * 100 for dp in ALL_DIPEPTIDES}


def _ctd_group_of(aa: str, groups: dict):
    for group_id, members in groups.items():
        if aa in members:
            return group_id
    return None


def ctd_features(seq: str) -> dict:
    seq = clean_sequence(seq)
    length = len(seq)
    feats = {}
    for attr, groups in CTD_GROUPS.items():
        group_seq = [_ctd_group_of(aa, groups) for aa in seq]
        counts = Counter(group_seq)
        for gid in (1, 2, 3):
            feats[f"CTD_{attr}_C{gid}"] = counts.get(gid, 0) / length * 100
        denom = max(length - 1, 1)
        transitions = Counter()
        for i in range(length - 1):
            left, right = group_seq[i], group_seq[i + 1]
            if left is None or right is None or left == right:
                continue
            transitions[tuple(sorted((left, right)))] += 1
        for pair in [(1, 2), (1, 3), (2, 3)]:
            feats[f"CTD_{attr}_T{pair[0]}{pair[1]}"] = transitions.get(pair, 0) / denom * 100
        for gid in (1, 2, 3):
            positions = [i + 1 for i, g in enumerate(group_seq) if g == gid]
            if not positions:
                distribution = [0.0] * 5
            else:
                n = len(positions)
                idxs = [1, max(1, math.ceil(0.25 * n)), max(1, math.ceil(0.50 * n)),
                        max(1, math.ceil(0.75 * n)), n]
                distribution = [positions[idx - 1] / length * 100 for idx in idxs]
            for pct, val in zip([0, 25, 50, 75, 100], distribution):
                feats[f"CTD_{attr}_D{gid}_{pct}"] = val
    return feats


def physicochemical_features(seq: str) -> dict:
    seq = clean_sequence(seq)
    feats = {}
    for name, table in [("Hydrophobicity_KD", kd), ("Polarity_Grantham", AA_POLARITY_GRANTHAM),
                         ("Volume_Zamyatnin", AA_VOLUME_ZAMYATNIN)]:
        values = np.array([table[aa] for aa in seq], dtype=float)
        feats[f"PHYS_{name}_mean"] = float(values.mean())
        feats[f"PHYS_{name}_std"] = float(values.std())
    analysis = ProteinAnalysis(seq)
    feats["PHYS_molecular_weight"] = analysis.molecular_weight()
    feats["PHYS_aromaticity"] = analysis.aromaticity()
    feats["PHYS_instability_index"] = analysis.instability_index()
    try:
        feats["PHYS_isoelectric_point"] = analysis.isoelectric_point()
    except Exception:
        feats["PHYS_isoelectric_point"] = np.nan
    feats["PHYS_gravy"] = analysis.gravy()
    feats["PHYS_charge_at_pH7"] = analysis.charge_at_pH(7.0)
    helix, turn, sheet = analysis.secondary_structure_fraction()
    feats["PHYS_ss_fraction_helix"] = helix
    feats["PHYS_ss_fraction_turn"] = turn
    feats["PHYS_ss_fraction_sheet"] = sheet
    feats["PHYS_length"] = len(seq)
    return feats

### Cell 6: Classical features, part 2 and feature_family

**What runs:** functions for DHKR, autocorrelation, BLOSUM, terminal composition and MHC-class-specific anchor features. At the end, `feature_family` decides from the column prefix which block (ESM2 / MHC / ANCHOR / classical) a column belongs to. The ablation block split depends on this. Definitions only.

In [ ]:
# ---- notebook cell 14 ----
def dhkr_features(seq: str) -> dict:
    seq = clean_sequence(seq)
    length = len(seq)
    counts = Counter(seq)
    feats = {f"DHKR_frac_{aa}": counts.get(aa, 0) / length * 100 for aa in "DHKRE"}
    feats["DHKR_charged_fraction"] = sum(counts.get(aa, 0) for aa in "DHKRE") / length * 100
    feats["DHKR_net_charge_proxy"] = (
        counts.get("K", 0) + counts.get("R", 0) + 0.1 * counts.get("H", 0)
        - counts.get("D", 0) - counts.get("E", 0)
    ) / length
    return feats


def _moran_autocorrelation(seq: str, table: dict, maxlag: int) -> list:
    values = np.array([table[aa] for aa in clean_sequence(seq)], dtype=float)
    length = len(values)
    mean_value = values.mean()
    denom = np.sum((values - mean_value) ** 2) / length
    out = []
    for lag in range(1, maxlag + 1):
        if lag >= length or denom == 0:
            out.append(0.0)
            continue
        numerator = np.sum((values[:-lag] - mean_value) * (values[lag:] - mean_value)) / (length - lag)
        out.append(float(numerator / denom))
    return out


def autocorrelation_features(seq: str, maxlag: int = 5) -> dict:
    seq = clean_sequence(seq)
    maxlag = max(0, min(maxlag, len(seq) - 1))
    feats = {}
    for name, table in [("Hydrophobicity_KD", kd), ("Polarity_Grantham", AA_POLARITY_GRANTHAM),
                         ("Volume_Zamyatnin", AA_VOLUME_ZAMYATNIN)]:
        for lag, value in enumerate(_moran_autocorrelation(seq, table, maxlag), start=1):
            feats[f"AUTOC_Moran_{name}_lag{lag}"] = value
    return feats


def blosum62_features(seq: str) -> dict:
    seq = clean_sequence(seq)
    acc = {aa: 0.0 for aa in STANDARD_AA}
    for residue in seq:
        for aa in STANDARD_AA:
            acc[aa] += BLOSUM62[residue][aa]
    return {f"BLOSUM_{aa}": acc[aa] / len(seq) for aa in STANDARD_AA}


def terminal_composition_features(seq: str) -> dict:
    seq = clean_sequence(seq)
    feats = {}
    for side, window in [("N", seq[:2]), ("C", seq[-2:]), ("N3", seq[:3]), ("C3", seq[-3:])]:
        counts = Counter(window)
        denom = max(len(window), 1)
        for aa in STANDARD_AA:
            feats[f"TERM_{side}_AAC_{aa}"] = counts.get(aa, 0) / denom * 100
    return feats


def anchor_position_features(seq: str, mhc_class: str) -> dict:

    seq = clean_sequence(seq)
    L = len(seq)
    mhc_class = str(mhc_class).strip().upper() if mhc_class is not None else ""
    feats = {
        "ANCHOR_P2_hydro": 0.0, "ANCHOR_Pomega_hydro": 0.0,
        "ANCHOR_P2_is_hydrophobic": 0.0, "ANCHOR_Pomega_is_hydrophobic": 0.0,
        "ANCHOR_classII_a1_hydro": 0.0, "ANCHOR_classII_a4_hydro": 0.0,
        "ANCHOR_classII_a6_hydro": 0.0, "ANCHOR_classII_a9_hydro": 0.0,
    }
    if L == 0:
        return feats
    if mhc_class == "I":
        p2 = seq[1] if L > 1 else seq[0]
        pomega = seq[-1]
        feats["ANCHOR_P2_hydro"] = float(kd.get(p2, 0.0))
        feats["ANCHOR_Pomega_hydro"] = float(kd.get(pomega, 0.0))
        feats["ANCHOR_P2_is_hydrophobic"] = float(p2 in HYDROPHOBIC_AA)
        feats["ANCHOR_Pomega_is_hydrophobic"] = float(pomega in HYDROPHOBIC_AA)
    elif mhc_class == "II":
        idxs = [max(0, min(L - 1, int(round(f * (L - 1))))) for f in (0.0, 0.33, 0.55, 0.9)]
        keys = ["ANCHOR_classII_a1_hydro", "ANCHOR_classII_a4_hydro",
                "ANCHOR_classII_a6_hydro", "ANCHOR_classII_a9_hydro"]
        for key, idx in zip(keys, idxs):
            feats[key] = float(kd.get(seq[idx], 0.0))
    return feats


def extract_all_features(seq: str, cfg: Config, mhc_class: str = None) -> dict:
    seq = clean_sequence(seq)
    feats = {}
    feats.update(aac_features(seq))
    if cfg.include_dpc:
        feats.update(dpc_features(seq))
    feats.update(ctd_features(seq))
    feats.update(physicochemical_features(seq))
    feats.update(dhkr_features(seq))
    feats.update(autocorrelation_features(seq, maxlag=cfg.autoc_maxlag))
    feats.update(blosum62_features(seq))
    feats.update(terminal_composition_features(seq))
    if cfg.include_anchor_features:
        feats.update(anchor_position_features(seq, mhc_class))
    return feats


def feature_family(feature_name: str) -> str:
    for prefix, fam in [("AAC_", "AAC"), ("DPC_", "DPC"), ("CTD_", "CTD"), ("PHYS_", "PHYS"),
                         ("DHKR_", "DHKR"), ("AUTOC_", "AUTOC"), ("BLOSUM_", "BLOSUM"),
                         ("TERM_", "TERMINAL"), ("ANCHOR_", "ANCHOR"),
                         ("ESM2_", "ESM2"), ("MHC_", "MHC")]:
        if feature_name.startswith(prefix):
            return fam
    return "OTHER"

### Cell 7: MHC features and ESM-2 embeddings

**What runs:** functions that build features from MHC percentile/IC50/allele and compute ESM-2 embeddings. Definitions only. The model download and the embedding computation happen in Cell 19.

In [ ]:
# ---- notebook cell 16 ----
def extract_mhc_features(df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    mhc = pd.DataFrame(index=df.index)
    rank = pd.to_numeric(df.get(cfg.mhc_rank_col), errors="coerce")
    ic50 = pd.to_numeric(df.get(cfg.mhc_ic50_col), errors="coerce")

    mhc["MHC_rank_missing"] = rank.isna().astype(float)
    mhc["MHC_ic50_missing"] = ic50.isna().astype(float)

    approx_rank_from_ic50 = np.clip(np.log10(ic50 + 1.0) * 2.5, 0, 100)
    approx_ic50_from_rank = np.clip(10 ** (rank / 2.5), 1, 100000)
    mhc["MHC_rank"] = rank.fillna(approx_rank_from_ic50).fillna(100.0)
    mhc["MHC_ic50"] = ic50.fillna(approx_ic50_from_rank).fillna(50000.0)
    mhc["MHC_log10_ic50"] = np.log10(mhc["MHC_ic50"] + 1.0)
    mhc["MHC_sqrt_rank"] = np.sqrt(mhc["MHC_rank"])

    mhc["MHC_strong_binder"] = ((mhc["MHC_rank"] < 0.5) | (mhc["MHC_ic50"] < 50.0)).astype(float)
    mhc["MHC_good_binder"] = (((mhc["MHC_rank"] >= 0.5) & (mhc["MHC_rank"] < 2.0)) |
                               ((mhc["MHC_ic50"] >= 50.0) & (mhc["MHC_ic50"] < 500.0))).astype(float)
    mhc["MHC_weak_binder"] = (((mhc["MHC_rank"] >= 2.0) & (mhc["MHC_rank"] < 10.0)) |
                               ((mhc["MHC_ic50"] >= 500.0) & (mhc["MHC_ic50"] < 5000.0))).astype(float)
    mhc["MHC_non_binder"] = ((mhc["MHC_rank"] >= 10.0) & (mhc["MHC_ic50"] >= 5000.0)).astype(float)
    mhc["MHC_binding_score"] = np.clip(1.0 / (1.0 + mhc["MHC_rank"]), 0, 1)
    mhc["MHC_log_ic50_score"] = np.clip(1.0 - (np.log10(mhc["MHC_ic50"] + 1) / 5.0), 0, 1)

    class_used = df.get(cfg.mhc_class_col, pd.Series("unknown", index=df.index)).astype(str).str.strip().str.upper()
    mhc["MHC_class_I"] = (class_used == "I").astype(float)
    mhc["MHC_class_II"] = (class_used == "II").astype(float)

    length = df[cfg.seq_col].astype(str).str.len()
    mhc["MHC_classI_x_short"] = mhc["MHC_class_I"] * ((length >= 8) & (length <= 11)).astype(float)
    mhc["MHC_classII_x_long"] = mhc["MHC_class_II"] * ((length >= 13) & (length <= 25)).astype(float)
    mhc["MHC_compatible_length"] = (
        mhc["MHC_class_I"] * ((length >= 8) & (length <= 11)).astype(float)
        + mhc["MHC_class_II"] * ((length >= 12) & (length <= 25)).astype(float)
    )

    if cfg.mhc_allele_col in df.columns:
        allele = df[cfg.mhc_allele_col].fillna("unknown").astype(str).str.strip()

        def allele_supertype(a):
            a = str(a).upper()
            for prefix, fam in [("HLA-A", "HLA_A"), ("HLA-B", "HLA_B"), ("HLA-C", "HLA_C"),
                                 ("HLA-DR", "HLA_DR"), ("HLA-DQ", "HLA_DQ"), ("HLA-DP", "HLA_DP")]:
                if a.startswith(prefix):
                    return fam
            return "Other"

        fam = allele.apply(allele_supertype)
        fam_counts = fam.value_counts()
        valid_fams = fam_counts[fam_counts >= cfg.mhc_allele_min_count].index.tolist()
        for f in valid_fams:
            mhc[f"MHC_allele_fam_{f}"] = (fam == f).astype(float)

    return mhc.fillna(0.0)


def _esm_cache_path(seqs: list, cfg: Config) -> Path:
    sig = f"{cfg.esm_model_name}|{cfg.esm_repr_layer}|{len(seqs)}|{seqs[0]}|{seqs[-1]}"
    h = hashlib.md5(sig.encode()).hexdigest()[:16]
    return Path(f".esm_cache_{h}.pkl")


def esm2_embedding_features(seqs: list, cfg: Config) -> pd.DataFrame:
    if not cfg.use_esm_embeddings:
        return pd.DataFrame(index=pd.RangeIndex(len(seqs)))
    if not HAS_ESM:
        raise RuntimeError(
            "ESM-2 requires 'torch' and 'fair-esm'. Install with:\n"
            "  pip install torch --index-url https://download.pytorch.org/whl/cpu\n"
            "  pip install fair-esm"
        )
    if not seqs:
        return pd.DataFrame(index=pd.RangeIndex(0))

    cache_path = _esm_cache_path(seqs, cfg)
    if cache_path.exists():
        print(f"[ESM-2] Loading cached embeddings from {cache_path}")
        return pd.read_pickle(cache_path)

    device = cfg.esm_device.strip() if cfg.esm_device else ("cuda" if torch.cuda.is_available() else "cpu")
    if not hasattr(esm.pretrained, cfg.esm_model_name):
        raise ValueError(f"Unknown ESM model '{cfg.esm_model_name}'.")

    print(f"[ESM-2] Loading {cfg.esm_model_name} (layer {cfg.esm_repr_layer}) on {device}.")
    model, alphabet = getattr(esm.pretrained, cfg.esm_model_name)()
    model = model.eval().to(device)
    batch_converter = alphabet.get_batch_converter()

    records, feature_names, total = [], None, len(seqs)
    with torch.no_grad():
        for start in range(0, total, cfg.esm_batch_size):
            batch_seqs = seqs[start:start + cfg.esm_batch_size]
            batch = [(str(start + i), seq) for i, seq in enumerate(batch_seqs)]
            _, strings, tokens = batch_converter(batch)
            tokens = tokens.to(device)
            output = model(tokens, repr_layers=[cfg.esm_repr_layer], return_contacts=False)
            reps = output["representations"][cfg.esm_repr_layer].detach().cpu().numpy()
            for row_idx, seq in enumerate(strings):
                vector = reps[row_idx, 1:len(seq) + 1].mean(axis=0)
                if feature_names is None:
                    feature_names = [f"ESM2_{i:03d}" for i in range(vector.shape[0])]
                records.append(vector.astype(float))
            print(f"[ESM-2] embedded {min(start + len(batch_seqs), total)}/{total} peptides")

    out_df = pd.DataFrame(records, columns=feature_names)
    out_df.to_pickle(cache_path)
    print(f"[ESM-2] Saved embeddings to {cache_path}")
    return out_df

### Cell 8: Feature matrix builder

**What runs:** a function that joins all features into one matrix. Definitions only.

In [ ]:
# ---- notebook cell 18 ----
def build_feature_matrix(df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    """Assumes df has already been filtered to valid sequences upstream."""
    clean_list = df[cfg.seq_col].astype(str).map(clean_sequence).tolist()
    if len(clean_list) == 0:
        raise ValueError("No peptides remain after upstream filtering.")

    if cfg.include_anchor_features and cfg.mhc_class_col in df.columns:
        mhc_class_list = df[cfg.mhc_class_col].astype(str).tolist()
    else:
        mhc_class_list = [None] * len(clean_list)

    records = [extract_all_features(seq, cfg, mhc_class=cls)
               for seq, cls in zip(clean_list, mhc_class_list)]
    feat_df = pd.DataFrame.from_records(records).fillna(0.0)

    esm_df = esm2_embedding_features(clean_list, cfg).fillna(0.0)
    feat_df = pd.concat([feat_df.reset_index(drop=True), esm_df.reset_index(drop=True)], axis=1)

    if cfg.use_mhc_features:
        mhc_df = extract_mhc_features(df, cfg).reset_index(drop=True)
        feat_df = pd.concat([feat_df, mhc_df], axis=1)
        print(f"[feature extraction] Added {mhc_df.shape[1]} MHC features.")

    lead_cols = {}
    if cfg.id_col and cfg.id_col in df.columns:
        lead_cols[cfg.id_col] = df[cfg.id_col].values
    lead_cols[cfg.label_col] = df[cfg.label_col].values
    lead_cols[cfg.seq_col] = df[cfg.seq_col].values

    print(f"[feature extraction] Built {feat_df.shape[1]} features "
          f"(DPC {'ON' if cfg.include_dpc else 'OFF'}, "
          f"anchor {'ON' if cfg.include_anchor_features else 'OFF'}) "
          f"for {feat_df.shape[0]} peptides.")
    return pd.concat([pd.DataFrame(lead_cols).reset_index(drop=True), feat_df.reset_index(drop=True)], axis=1)

### Cell 9: Assay filter and label-conflict resolution

**What runs:** functions that restrict the data to T-cell assays and resolve peptides with contradictory labels. Definitions only.

In [ ]:
# ---- notebook cell 20 ----
def restrict_to_assay_supported(df: pd.DataFrame, cfg: Config) -> tuple:
    n_before = len(df)
    keep = pd.Series(True, index=df.index)
    reasons = {}

    if cfg.require_mhc_assay:
        rank = pd.to_numeric(df.get(cfg.mhc_rank_col), errors="coerce")
        ic50 = pd.to_numeric(df.get(cfg.mhc_ic50_col), errors="coerce")
        has_assay = rank.notna() | ic50.notna()
        reasons["dropped_no_assay_value"] = int((~has_assay).sum())
        keep &= has_assay

    if cfg.require_known_class:
        class_used = df.get(cfg.mhc_class_col, pd.Series(index=df.index, dtype=object))
        class_used = class_used.astype(str).str.strip().str.upper()
        has_class = class_used.isin(["I", "II"])
        reasons["dropped_unknown_class"] = int((~has_class & keep).sum())
        keep &= has_class

    out = df.loc[keep].reset_index(drop=True)
    report = {"n_before": n_before, "n_after": int(len(out)), **reasons}
    print(f"[restrict] {n_before} -> {len(out)} peptides after MHC assay-support restriction: {reasons}")
    return out, report


def restrict_to_tcell_assay(df: pd.DataFrame, cfg: Config) -> tuple:

    if not cfg.require_tcell_assay or cfg.assay_col not in df.columns:
        return df, {"require_tcell_assay": False}
    n_before = len(df)
    has_tcell = df[cfg.assay_col].astype(str).str.contains(cfg.tcell_assay_keyword, na=False, case=False)
    out = df.loc[has_tcell].reset_index(drop=True)
    report = {
        "n_before_tcell_filter": n_before,
        "n_after_tcell_filter": int(len(out)),
        "n_dropped_no_tcell_evidence": int((~has_tcell).sum()),
    }
    print(f"[restrict] T-cell-assay filter: {n_before} -> {len(out)} peptides "
          f"({report['n_dropped_no_tcell_evidence']} dropped -- no direct T-cell assay evidence).")
    return out, report


def resolve_label_conflicts(df: pd.DataFrame, groups: np.ndarray, cfg: Config) -> tuple:
    y = df[cfg.label_col].to_numpy().astype(int)
    resolved = y.copy()
    drop_mask = np.zeros(len(df), dtype=bool)
    n_relabeled_groups, n_relabeled_peptides = 0, 0
    n_dropped_groups, n_dropped_peptides = 0, 0

    for g in np.unique(groups):
        idx = np.where(groups == g)[0]
        if len(idx) <= 1:
            continue
        labels = y[idx]
        if len(set(labels.tolist())) == 1:
            continue
        pos_frac = float((labels == 1).mean())
        if pos_frac >= cfg.conflict_consensus_fraction:
            changed = idx[labels == 0]
            resolved[idx] = 1
            n_relabeled_groups += 1
            n_relabeled_peptides += len(changed)
        elif pos_frac <= (1.0 - cfg.conflict_consensus_fraction):
            changed = idx[labels == 1]
            resolved[idx] = 0
            n_relabeled_groups += 1
            n_relabeled_peptides += len(changed)
        else:
            drop_mask[idx] = True
            n_dropped_groups += 1
            n_dropped_peptides += len(idx)

    out = df.copy()
    out[cfg.label_col] = resolved
    out = out.loc[~drop_mask].reset_index(drop=True)
    kept_groups = groups[~drop_mask]
    report = {
        "n_conflicting_groups_relabeled": n_relabeled_groups,
        "n_peptides_relabeled_to_consensus": n_relabeled_peptides,
        "n_conflicting_groups_dropped": n_dropped_groups,
        "n_peptides_dropped_as_contradictory": n_dropped_peptides,
        "n_peptides_after_resolution": int(len(out)),
    }
    print(f"[conflict resolution] {report}")
    return out, kept_groups, report

### Cell 10: Feature selectors and ensemble

**What runs:** the variance/correlation filter, the mutual-information top-k selector and the weighted soft-voting ensemble classes. Definitions only.

In [ ]:
# ---- notebook cell 22 ----
def remove_low_variance(X_train: pd.DataFrame, threshold: float) -> list:
    variances = X_train.var(axis=0)
    return variances[variances > threshold].index.tolist()


def remove_highly_correlated(X_train: pd.DataFrame, threshold: float) -> list:
    corr = X_train.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
    to_drop = {col for col in upper.columns if any(upper[col] > threshold)}
    return [c for c in X_train.columns if c not in to_drop]


class VarianceCorrelationFilter(BaseEstimator, TransformerMixin):
    def __init__(self, var_threshold: float = 1e-5, corr_threshold: float = 0.90):
        self.var_threshold = var_threshold
        self.corr_threshold = corr_threshold

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X).copy()
        self.feature_names_in_ = X_df.columns.astype(str).tolist()
        X_df.columns = self.feature_names_in_
        cols_var = remove_low_variance(X_df, self.var_threshold)
        self.selected_columns_ = remove_highly_correlated(X_df[cols_var], self.corr_threshold)
        if not self.selected_columns_:
            self.selected_columns_ = cols_var[:1] if cols_var else self.feature_names_in_[:1]
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X).copy()
        X_df.columns = self.feature_names_in_
        return X_df[self.selected_columns_]


class MutualInfoTopKSelector(BaseEstimator, TransformerMixin):
    def __init__(self, k: int = 250, random_state: int = 42, force_include_prefix: str = "MHC_"):
        self.k = k
        self.random_state = random_state
        self.force_include_prefix = force_include_prefix

    def fit(self, X, y):
        X_df = pd.DataFrame(X).copy()
        self.feature_names_in_ = X_df.columns.astype(str).tolist()
        X_df.columns = self.feature_names_in_
        forced = [c for c in self.feature_names_in_ if c.startswith(self.force_include_prefix)]
        remaining = [c for c in self.feature_names_in_ if c not in forced]
        if remaining and self.k and self.k > 0:
            n_from_mi = max(0, self.k - len(forced))
            if 0 < n_from_mi < len(remaining):
                mi = mutual_info_classif(X_df[remaining], y, random_state=self.random_state)
                order = np.argsort(mi)[::-1][:n_from_mi]
                selected_from_mi = [remaining[i] for i in order]
            else:
                selected_from_mi = remaining
        else:
            selected_from_mi = remaining
        self.selected_columns_ = forced + selected_from_mi
        if not self.selected_columns_:
            self.selected_columns_ = self.feature_names_in_
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X).copy()
        X_df.columns = self.feature_names_in_
        return X_df[self.selected_columns_]


class WeightedSoftVotingEnsemble(BaseEstimator, ClassifierMixin):
    def __init__(self, estimators: dict, weights: np.ndarray, threshold: float = 0.5):
        self.estimators = estimators
        self.weights = weights
        self.threshold = threshold
        self.classes_ = np.array([0, 1])

    def fit(self, X, y=None):
        return self

    def predict_proba(self, X):
        probs = np.array([est.predict_proba(X)[:, 1] for est in self.estimators.values()])
        pos = np.average(probs, axis=0, weights=self.weights)
        return np.column_stack([1.0 - pos, pos])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= self.threshold).astype(int)


def make_model_pipeline(estimator, cfg: Config) -> Pipeline:
    return Pipeline(steps=[
        ("var_corr", VarianceCorrelationFilter(cfg.var_threshold, cfg.corr_threshold)),
        ("mi", MutualInfoTopKSelector(k=cfg.mi_top_k, random_state=cfg.random_seed)),
        ("scaler", StandardScaler()),
        ("classifier", estimator),
    ])


def pipeline_selected_features(estimator) -> list:
    if not isinstance(estimator, Pipeline):
        return []
    try:
        return estimator.named_steps["mi"].selected_columns_
    except Exception:
        return []


def optimize_threshold(y_true: np.ndarray, proba: np.ndarray, metric: str) -> float:
    precision, recall, thresholds = precision_recall_curve(y_true, proba)
    if len(thresholds) == 0:
        return 0.5
    candidates = np.unique(np.concatenate([thresholds, np.linspace(0.10, 0.90, 81)]))
    best_threshold, best_score = 0.5, -np.inf
    for threshold in candidates:
        pred = (proba >= threshold).astype(int)
        if metric == "f1":
            score = f1_score(y_true, pred, zero_division=0)
        elif metric == "balanced_accuracy":
            score = balanced_accuracy_score(y_true, pred)
        else:
            score = matthews_corrcoef(y_true, pred)
        if score > best_score:
            best_score, best_threshold = score, float(threshold)
    return best_threshold

### Cell 11: Hyper-parameter search

**What runs:** the search space of each model and the tuning functions. Definitions only.

In [ ]:
# ---- notebook cell 24 ----
def model_search_space(y_train: np.ndarray, cfg: Config) -> dict:
    n_pos, n_neg = int((y_train == 1).sum()), int((y_train == 0).sum())
    scale_pos_weight = n_neg / max(n_pos, 1)
    space = {
        "logreg_l2": (make_model_pipeline(
            LogisticRegression(penalty="l2", solver="lbfgs", max_iter=6000,
                                class_weight="balanced", random_state=cfg.random_seed), cfg),
            {"classifier__C": loguniform(1e-3, 1e2)}),
        "svm_rbf": (make_model_pipeline(
            SVC(kernel="rbf", probability=True, class_weight="balanced",
                random_state=cfg.random_seed, cache_size=1000), cfg),
            {"classifier__C": loguniform(1e-2, 1e2), "classifier__gamma": loguniform(1e-4, 1e0)}),
        "nu_svm": (make_model_pipeline(
            NuSVC(kernel="rbf", gamma="scale", probability=True,
                  random_state=cfg.random_seed, cache_size=1000), cfg),
            {"classifier__nu": [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8],
             "classifier__gamma": loguniform(1e-4, 1e0)}),
        "rf": (make_model_pipeline(
            RandomForestClassifier(random_state=cfg.random_seed, n_jobs=1,
                                    class_weight="balanced_subsample"), cfg),
            {"classifier__n_estimators": randint(300, 900), "classifier__max_depth": [3, 5, 8, 12, 16, None],
             "classifier__min_samples_leaf": randint(1, 10), "classifier__min_samples_split": randint(2, 16),
             "classifier__max_features": ["sqrt", "log2", 0.25, 0.4, 0.6]}),
        "extra_trees": (make_model_pipeline(
            ExtraTreesClassifier(random_state=cfg.random_seed, n_jobs=1, class_weight="balanced"), cfg),
            {"classifier__n_estimators": randint(300, 900), "classifier__max_depth": [3, 5, 8, 12, 16, None],
             "classifier__min_samples_leaf": randint(1, 8), "classifier__min_samples_split": randint(2, 14),
             "classifier__max_features": ["sqrt", "log2", 0.25, 0.4, 0.6]}),
    }

    hist_kwargs = {"random_state": cfg.random_seed}
    if _HISTGB_HAS_CLASS_WEIGHT:
        hist_kwargs["class_weight"] = "balanced"

    space["hist_gb"] = (make_model_pipeline(
        HistGradientBoostingClassifier(**hist_kwargs), cfg),
        {"classifier__max_iter": randint(100, 500), "classifier__max_leaf_nodes": randint(7, 63),
         "classifier__learning_rate": loguniform(0.01, 0.25),
         "classifier__l2_regularization": loguniform(1e-4, 20),
         "classifier__min_samples_leaf": randint(10, 60)})

    if HAS_XGB:
        space["xgb"] = (make_model_pipeline(
            XGBClassifier(random_state=cfg.random_seed, n_jobs=1, eval_metric="auc",
                        tree_method="hist", scale_pos_weight=scale_pos_weight), cfg),
            {"classifier__n_estimators": randint(150, 700), "classifier__max_depth": randint(2, 7),
             "classifier__learning_rate": loguniform(0.01, 0.25), "classifier__subsample": [0.60, 0.75, 0.90, 1.0],
             "classifier__colsample_bytree": [0.50, 0.65, 0.80, 1.0], "classifier__min_child_weight": randint(1, 12),
             "classifier__reg_alpha": loguniform(1e-4, 10), "classifier__reg_lambda": loguniform(1e-3, 30)})
    return space


def tune_models(X_train, y_train, inner_cv, cfg: Config, groups_train=None) -> dict:
    tuned = {}
    for name, (estimator, params) in model_search_space(y_train, cfg).items():
        search = RandomizedSearchCV(estimator=estimator, param_distributions=params, n_iter=cfg.random_search_iter,
                                     scoring="roc_auc", cv=inner_cv, random_state=cfg.random_seed,
                                     n_jobs=cfg.n_jobs, refit=True, error_score=np.nan)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            if groups_train is None:
                search.fit(X_train, y_train)
            else:
                search.fit(X_train, y_train, groups=groups_train)
        best_estimator = search.best_estimator_
        try:
            cv_kwargs = {"groups": groups_train} if groups_train is not None else {}
            oof_proba = cross_val_predict(clone(best_estimator), X_train, y_train, cv=inner_cv,
                                           method="predict_proba", n_jobs=cfg.n_jobs, **cv_kwargs)[:, 1]
            threshold = optimize_threshold(y_train, oof_proba, cfg.threshold_metric)
        except Exception:
            threshold = 0.5
        tuned[name] = {
            "estimator": best_estimator, "inner_cv_auc": float(search.best_score_),
            "best_params": search.best_params_, "threshold": threshold,
            "selected_features": pipeline_selected_features(best_estimator),
        }
    return tuned


def ensemble_member_items(tuned: dict, min_inner_auc: float) -> list:
    eligible = [(n, i) for n, i in tuned.items() if np.isfinite(i["inner_cv_auc"]) and i["inner_cv_auc"] >= min_inner_auc]
    if eligible:
        return eligible
    finite = [(n, i) for n, i in tuned.items() if np.isfinite(i["inner_cv_auc"])]
    return [max(finite, key=lambda item: item[1]["inner_cv_auc"])] if finite else list(tuned.items())


def ensemble_weights(member_items: list) -> np.ndarray:
    weights = np.array([max(info["inner_cv_auc"], 1e-6) for _, info in member_items], dtype=float)
    return weights / weights.sum()


def soft_vote_proba(tuned: dict, X: pd.DataFrame, min_inner_auc: float) -> np.ndarray:
    members = ensemble_member_items(tuned, min_inner_auc)
    weights = ensemble_weights(members)
    probs = np.array([info["estimator"].predict_proba(X)[:, 1] for _, info in members])
    return np.average(probs, axis=0, weights=weights)


def metric_row(y_true: np.ndarray, proba: np.ndarray, threshold: float) -> dict:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=UserWarning)
        pred = (proba >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
        specificity = tn / (tn + fp) if (tn + fp) else 0.0
        roc_auc = roc_auc_score(y_true, proba) if len(np.unique(y_true)) == 2 else np.nan
        pr_auc = average_precision_score(y_true, proba) if (y_true == 1).any() else np.nan
        result = {"roc_auc": roc_auc, "pr_auc": pr_auc, "accuracy": accuracy_score(y_true, pred),
                  "f1": f1_score(y_true, pred, zero_division=0), "mcc": matthews_corrcoef(y_true, pred),
                  "sensitivity": sensitivity, "specificity": specificity,
                  "balanced_accuracy": balanced_accuracy_score(y_true, pred), "threshold": threshold,
                  "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn)}
    return result

### Cell 12: Nested CV

**What runs:** defines `run_nested_cv`. The `CHANGED (R2.5/R3.1)` ensemble-threshold line is in this cell. Definitions only. The actual run happens in Cell 21.

In [ ]:
# ---- notebook cell 26 ----
def make_outer_splits(y: np.ndarray, groups: np.ndarray, cfg: Config):
    use_groups = cfg.use_similarity_cv in ("always", "auto")
    if not use_groups:
        splitter = RepeatedStratifiedKFold(n_splits=cfg.outer_folds, n_repeats=cfg.outer_repeats,
                                            random_state=cfg.random_seed)
        for split_id, (train_idx, val_idx) in enumerate(splitter.split(np.zeros(len(y)), y), start=1):
            repeat = (split_id - 1) // cfg.outer_folds + 1
            fold = (split_id - 1) % cfg.outer_folds + 1
            yield split_id, repeat, fold, train_idx, val_idx, "repeated_stratified_kfold"
        return

    n_unique_groups = len(np.unique(groups)) if groups is not None else 0
    effective_splits = min(cfg.outer_folds, n_unique_groups)
    if effective_splits < 2:
        raise ValueError(
            f"Cannot run group-aware CV: only {n_unique_groups} unique similarity "
            f"group(s) in this data but need >=2. Either this subset is too small/"
            f"too similarity-collapsed for group-aware CV, or set "
            f"use_similarity_cv='never' deliberately for this specific probe."
        )
    if effective_splits < cfg.outer_folds:
        print(f"[cv] WARNING: only {n_unique_groups} unique similarity groups available; "
              f"reducing outer_folds from {cfg.outer_folds} to {effective_splits} for this "
              f"run to avoid empty validation folds.")

    try:
        for repeat in range(1, cfg.outer_repeats + 1):
            splitter = StratifiedGroupKFold(n_splits=effective_splits, shuffle=True,
                                             random_state=cfg.random_seed + repeat - 1)
            for fold, (train_idx, val_idx) in enumerate(splitter.split(np.zeros(len(y)), y, groups=groups), start=1):
                if len(val_idx) == 0 or len(train_idx) == 0:
                    print(f"[cv] Skipping degenerate fold (repeat {repeat}, fold {fold}) "
                          f"with an empty train/val partition.")
                    continue
                split_id = (repeat - 1) * effective_splits + fold
                yield split_id, repeat, fold, train_idx, val_idx, "stratified_group_kfold_similarity_aware"
    except ValueError:
        raise
    except Exception as exc:
        print(f"[cv] StratifiedGroupKFold unavailable/failed ({exc}); falling back to GroupKFold.")
        splitter = GroupKFold(n_splits=effective_splits)
        for fold, (train_idx, val_idx) in enumerate(splitter.split(np.zeros(len(y)), y, groups=groups), start=1):
            if len(val_idx) == 0 or len(train_idx) == 0:
                continue
            yield fold, 1, fold, train_idx, val_idx, "group_kfold_similarity_aware"


def make_inner_cv(y_train: np.ndarray, groups_train: np.ndarray, cfg: Config, cv_mode: str):
    class_counts = np.bincount(y_train.astype(int), minlength=2)
    n_splits = min(cfg.inner_folds, int(class_counts.min()))
    if n_splits < 2:
        raise ValueError("Inner CV needs >=2 samples per class in the outer-training fold.")
    if "group" in cv_mode and groups_train is not None and len(np.unique(groups_train)) >= n_splits:
        sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=cfg.random_seed)
        try:
            next(sgkf.split(np.zeros(len(y_train)), y_train, groups=groups_train))
            return sgkf, groups_train
        except Exception as exc:
            print(f"[cv] StratifiedGroupKFold failed for inner CV ({exc}); falling back to StratifiedKFold.")
    return StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=cfg.random_seed), None


def run_nested_cv(X: pd.DataFrame, y: np.ndarray, groups: np.ndarray, cfg: Config,
                   row_ids: np.ndarray = None) -> tuple:
    if row_ids is None:
        row_ids = np.arange(len(y))

    rows, full_records, oof_records = [], [], []
    for split_id, repeat, fold, train_idx, val_idx, cv_mode in make_outer_splits(y, groups, cfg):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        groups_train = groups[train_idx] if groups is not None else None
        inner_cv, inner_groups = make_inner_cv(y_train, groups_train, cfg, cv_mode)
        tuned = tune_models(X_train, y_train, inner_cv, cfg, groups_train=inner_groups)

        split_rows = []
        for model_name, info in tuned.items():
            proba = info["estimator"].predict_proba(X_val)[:, 1]
            metrics = metric_row(y_val, proba, info["threshold"])
            row = {"split": split_id, "repeat": repeat, "fold": fold, "cv_mode": cv_mode, "model": model_name,
                   "n_features_final": len(info["selected_features"]), "inner_cv_auc": info["inner_cv_auc"], **metrics}
            split_rows.append(row)
            full_records.append({"row": row, "selected_features": info["selected_features"],
                                  "best_params": info["best_params"]})
            for i, ridx in enumerate(val_idx):
                oof_records.append({"row_id": row_ids[ridx], "repeat": repeat, "model": model_name,
                                     "proba": float(proba[i]), "y_true": int(y_val[i])})

        members = ensemble_member_items(tuned, cfg.ensemble_min_inner_auc)
        ens_proba = soft_vote_proba(tuned, X_val, cfg.ensemble_min_inner_auc)
        # CHANGED (R2.5/R3.1): was  optimize_threshold(y_val, ens_proba, cfg.threshold_metric)  which tuned the
        # threshold on the OUTER validation labels. Now: mean of the members' inner-CV thresholds, i.e. the same
        # rule that gives the final model's threshold. Only training-fold labels are used.
        ens_threshold = float(np.mean([info["threshold"] for _, info in members]))
        ens_metrics = metric_row(y_val, ens_proba, ens_threshold)
        ens_selected = sorted({f for _, info in members for f in info["selected_features"]})
        ens_row = {"split": split_id, "repeat": repeat, "fold": fold, "cv_mode": cv_mode, "model": "weighted_ensemble",
                   "n_features_final": len(ens_selected), "inner_cv_auc": np.mean([i["inner_cv_auc"] for _, i in members]),
                   "ensemble_members": ",".join(n for n, _ in members), **ens_metrics}
        split_rows.append(ens_row)
        full_records.append({"row": ens_row, "selected_features": ens_selected,
                              "best_params": {"ensemble_members": [n for n, _ in members]}})
        for i, ridx in enumerate(val_idx):
            oof_records.append({"row_id": row_ids[ridx], "repeat": repeat, "model": "weighted_ensemble",
                                 "proba": float(ens_proba[i]), "y_true": int(y_val[i])})
        rows.extend(split_rows)

        auc_text = " | ".join(f"{r['model']}={r['roc_auc']:.3f}" for r in split_rows
                               if r["model"] in ("rf", "extra_trees", "xgb", "weighted_ensemble"))
        print(f"[repeat {repeat} fold {fold}] features={ens_row['n_features_final']} | {auc_text}")
    return pd.DataFrame(rows), full_records, pd.DataFrame(oof_records)

### Cell 13: Summary helpers

**What runs:** confidence-interval and metric-summary functions. Definitions only.

In [ ]:
# ---- notebook cell 28 ----
def confidence_interval(values, confidence: float = 0.95) -> tuple:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan, np.nan
    mean = float(np.mean(values))
    if len(values) == 1:
        return mean, np.nan, np.nan
    sem = np.std(values, ddof=1) / math.sqrt(len(values))
    margin = t.ppf((1 + confidence) / 2, df=len(values) - 1) * sem
    return mean, float(mean - margin), float(mean + margin)


def summarize_metrics(cv_df: pd.DataFrame, outdir: Path) -> pd.DataFrame:
    metric_cols = ["roc_auc", "pr_auc", "f1", "mcc", "sensitivity", "specificity", "balanced_accuracy"]
    rows = []
    for model_name, group in cv_df.groupby("model"):
        row = {"model": model_name, "n_splits": int(len(group))}
        for metric in metric_cols:
            mean, low, high = confidence_interval(group[metric].to_numpy())
            row[f"{metric}_mean"], row[f"{metric}_ci95_low"], row[f"{metric}_ci95_high"] = mean, low, high
            row[f"{metric}_std"] = float(group[metric].std())
        rows.append(row)
    summary = pd.DataFrame(rows).sort_values("roc_auc_mean", ascending=False)
    summary.to_csv(outdir / "metric_summary_ci95.csv", index=False)
    return summary


def feature_selection_stability(full_records: list, outdir: Path) -> pd.DataFrame:
    main_records = [r for r in full_records if r["row"]["model"] != "weighted_ensemble"]
    all_feats = sorted({f for r in main_records for f in r["selected_features"]})
    counts = pd.Series(0, index=all_feats, dtype=int)
    for record in main_records:
        counts.loc[record["selected_features"]] += 1
    denom = max(len(main_records), 1)
    stability = (counts / denom).sort_values(ascending=False).rename("selection_frequency").to_frame()
    stability["family"] = [feature_family(f) for f in stability.index]
    stability.index.name = "feature"
    stability.to_csv(outdir / "feature_selection_stability.csv")
    return stability


def subgroup_performance_report(df: pd.DataFrame, oof_df: pd.DataFrame,
                                 subgroup_cols=("MHC_Class_Used", "Organism"),
                                 model_name: str = "weighted_ensemble",
                                 min_group_size: int = 10) -> pd.DataFrame:
    sub = oof_df[oof_df["model"] == model_name].copy()
    agg = sub.groupby("row_id").agg(proba=("proba", "mean"), y_true=("y_true", "first"),
                                     n_folds=("proba", "size")).reset_index()
    meta = df.reset_index(drop=True).reset_index().rename(columns={"index": "row_id"})
    keep_cols = ["row_id"] + [c for c in subgroup_cols if c in meta.columns]
    merged = agg.merge(meta[keep_cols], on="row_id", how="left")

    rows = []
    for col in subgroup_cols:
        if col not in merged.columns:
            continue
        for val, g in merged.groupby(col):
            if g["y_true"].nunique() < 2 or len(g) < min_group_size:
                continue
            rows.append({"subgroup_column": col, "subgroup_value": val, "n": int(len(g)),
                         "n_positive": int((g["y_true"] == 1).sum()),
                         "n_negative": int((g["y_true"] == 0).sum()),
                         "roc_auc": float(roc_auc_score(g["y_true"], g["proba"]))})
    out = pd.DataFrame(rows).sort_values(["subgroup_column", "roc_auc"], ascending=[True, False])
    return out


def quick_leakage_ablation(X: pd.DataFrame, y: np.ndarray, groups: np.ndarray, cfg: Config) -> dict:

    probe_kwargs = {**asdict(cfg), "random_search_iter": 10, "outer_repeats": 1}

    print("[leakage ablation] Running HONEST (similarity-group-aware) CV probe...")
    honest_cfg = Config(**{**probe_kwargs, "use_similarity_cv": "always"})
    honest_df, _, _ = run_nested_cv(X, y, groups, honest_cfg)
    honest_auc = float(honest_df.loc[honest_df["model"] == "weighted_ensemble", "roc_auc"].mean())

    print("[leakage ablation] Running NAIVE (non-grouped) CV probe -- for reporting the gap ONLY...")
    naive_cfg = Config(**{**probe_kwargs, "use_similarity_cv": "never"})
    naive_df, _, _ = run_nested_cv(X, y, groups, naive_cfg)
    naive_auc = float(naive_df.loc[naive_df["model"] == "weighted_ensemble", "roc_auc"].mean())

    result = {
        "honest_group_aware_auc_probe": honest_auc,
        "naive_non_grouped_auc_probe": naive_auc,
        "estimated_leakage_inflation": naive_auc - honest_auc,
        "note": "These are LIGHT-BUDGET probes (random_search_iter=10, outer_repeats=1) for "
                "quantifying leakage only. Report cv_fold_results.csv / metric_summary_ci95.csv "
                "(full-budget, similarity-group-aware) as the model's actual performance.",
    }
    print(f"[leakage ablation] {result}")
    return result

### Cell 14: block split and ablation functions

**What runs:** `BLOCK_FAMILIES` defines the 3 blocks (ESM2, MHC_ANCHOR, CLASSICAL) and `ABLATION_CONFIGS` defines the 7 configurations. It also defines `run_block_ablation` (a full tuned nested CV for each configuration), `run_fixed_et_ablation` (the fast mode: one fixed Extra Trees model on the same outer folds, folds run in parallel), the Nadeau-Bengio paired test and the summary-table functions. Definitions only.

In [ ]:
from scipy import stats
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import Ridge
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import CCA
from sklearn.metrics import roc_auc_score

BLOCK_FAMILIES = {
    "ESM2":       {"ESM2"},
    "MHC_ANCHOR": {"MHC", "ANCHOR"},
    "CLASSICAL":  {"AAC", "DPC", "CTD", "PHYS", "DHKR", "AUTOC", "BLOSUM", "TERMINAL", "OTHER"},
}

ABLATION_CONFIGS = {
    "MHC_ANCHOR only":        ["MHC_ANCHOR"],
    "ESM2 only":              ["ESM2"],
    "CLASSICAL only":         ["CLASSICAL"],
    "MHC_ANCHOR + ESM2":      ["MHC_ANCHOR", "ESM2"],        # = full minus CLASSICAL
    "MHC_ANCHOR + CLASSICAL": ["MHC_ANCHOR", "CLASSICAL"],   # = full minus ESM2
    "ESM2 + CLASSICAL":       ["ESM2", "CLASSICAL"],         # = full minus MHC_ANCHOR
    "FULL":                   ["MHC_ANCHOR", "ESM2", "CLASSICAL"],
}


def split_blocks(X):
    cols = {b: [c for c in X.columns if feature_family(c) in fams] for b, fams in BLOCK_FAMILIES.items()}
    n = sum(len(v) for v in cols.values())
    assert n == X.shape[1], f"{X.shape[1] - n} columns are not assigned to any block"
    return cols


def safe_name(s):
    return s.replace(" ", "_").replace("+", "plus")

# ---------------------------------------------------------------- R1.3 ----------------------------
def run_block_ablation(X, y, groups, cfg, outdir, which=None, skip_done=True):
    cols = split_blocks(X)
    print("[ablation] block sizes:", {b: len(c) for b, c in cols.items()})
    chosen = {k: ABLATION_CONFIGS[k] for k in (list(ABLATION_CONFIGS) if which is None else which) if k in ABLATION_CONFIGS}   # runs in the order given in `which`
    results = []
    for name, blocks in chosen.items():
        use = [c for b in blocks for c in cols[b]]
        done = Path(outdir) / f"ablation_cv_{safe_name(name)}.csv"
        if skip_done and done.exists():
            print(f"[ablation] {name}: found {done.name}, skipping (use --overwrite to recompute)")
            results.append(pd.read_csv(done)); continue
        c2 = type(cfg)(**{**asdict(cfg), "mi_top_k": min(cfg.mi_top_k, len(use))})
        print(f"\n[ablation] === {name}: {len(use)} features, mi_top_k={c2.mi_top_k}, "
              f"search_iter={c2.random_search_iter}, repeats={c2.outer_repeats} ===")
        t0 = time.time()
        cv_df, _, oof_df = run_nested_cv(X[use], y, groups, c2)
        cv_df["config"] = name; oof_df["config"] = name
        cv_df.to_csv(Path(outdir) / f"ablation_cv_{safe_name(name)}.csv", index=False)
        oof_df.to_csv(Path(outdir) / f"ablation_oof_{safe_name(name)}.csv", index=False)
        results.append(cv_df)
        e = cv_df[cv_df.model == "weighted_ensemble"]
        print(f"[ablation] {name}: ensemble ROC-AUC {e.roc_auc.mean():.3f} +/- {e.roc_auc.std():.3f} "
              f"({(time.time() - t0) / 60:.1f} min)")
    return pd.concat(results, ignore_index=True)


def load_ablation_results(outdir):
    files = [f for f in sorted(Path(outdir).glob("ablation_cv_*.csv")) if f.name != "ablation_cv_all_configs.csv"]
    if not files: raise SystemExit(f"No ablation_cv_<config>.csv files found in {outdir}")
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)


def nb_paired(a, b, folds=5):
    """Nadeau-Bengio corrected paired t-test on matched outer folds."""
    d = np.asarray(a, float) - np.asarray(b, float); J = len(d)
    se = d.std(ddof=1) * np.sqrt(1 / J + 1 / (folds - 1))
    tval = d.mean() / se; p = 2 * stats.t.sf(abs(tval), J - 1)
    h = stats.t.ppf(.975, J - 1) * se
    return d.mean(), d.mean() - h, d.mean() + h, p


def summarize_ablation(all_cv, folds=5, model="weighted_ensemble", metric="roc_auc"):
    d = all_cv[all_cv.model == model].pivot_table(index="split", columns="config", values=metric)
    J = len(d)
    rows = []
    for c in d.columns:
        sd = d[c].std(); se = sd * np.sqrt(1 / J + 1 / (folds - 1)); h = stats.t.ppf(.975, J - 1) * se
        row = {"config": c, f"{metric}_mean": d[c].mean(), "sd_across_folds": sd,
               "corrected_ci_lo": d[c].mean() - h, "corrected_ci_hi": d[c].mean() + h}
        if "FULL" in d.columns and c != "FULL":
            diff = nb_paired(d[c], d["FULL"], folds)
            row.update({"delta_vs_FULL": diff[0], "delta_ci_lo": diff[1], "delta_ci_hi": diff[2], "p_corrected": diff[3]})
        rows.append(row)
    table = pd.DataFrame(rows).sort_values(f"{metric}_mean", ascending=False)
    pairs = {
        "adding ESM2 to MHC_ANCHOR":                ("MHC_ANCHOR + ESM2", "MHC_ANCHOR only"),
        "adding CLASSICAL to MHC_ANCHOR":           ("MHC_ANCHOR + CLASSICAL", "MHC_ANCHOR only"),
        "adding ESM2 to MHC_ANCHOR + CLASSICAL":    ("FULL", "MHC_ANCHOR + CLASSICAL"),
        "adding MHC_ANCHOR to ESM2 + CLASSICAL":    ("FULL", "ESM2 + CLASSICAL"),
        "adding CLASSICAL to ESM2":                 ("ESM2 + CLASSICAL", "ESM2 only"),
        "adding MHC_ANCHOR to CLASSICAL":           ("MHC_ANCHOR + CLASSICAL", "CLASSICAL only"),
    }
    inc = {}
    for label, (a, b) in pairs.items():
        if a in d.columns and b in d.columns:
            inc[label] = nb_paired(d[a], d[b], folds)
    inc = pd.DataFrame(inc, index=["delta", "ci_lo", "ci_hi", "p_corrected"]).T if inc else pd.DataFrame()
    return table, inc


# ---- fast mode: one fixed Extra Trees learner, same outer folds as the main analysis ----
from joblib import Parallel, delayed
from sklearn.metrics import roc_auc_score, average_precision_score

def _et_one_fold(Xu, y, tr, va, cfg_, n_trees, seed):
    pipe = make_model_pipeline(ExtraTreesClassifier(n_estimators=n_trees, class_weight="balanced",
                                                    random_state=seed, n_jobs=1), cfg_)
    pipe.fit(Xu.iloc[tr], y[tr])
    p = pipe.predict_proba(Xu.iloc[va])[:, 1]
    return roc_auc_score(y[va], p), average_precision_score(y[va], p), len(pipe.named_steps["mi"].selected_columns_)

def run_fixed_et_ablation(X, y, groups, cfg, outdir, which=None, skip_done=True, n_trees=500):
    cols = split_blocks(X)
    print("[ablation-ET] block sizes:", {b: len(c) for b, c in cols.items()})
    chosen = {k: ABLATION_CONFIGS[k] for k in (list(ABLATION_CONFIGS) if which is None else which) if k in ABLATION_CONFIGS}   # runs in the order given in `which`
    splits = list(make_outer_splits(y, groups, cfg))   # identical folds to the main analysis
    print(f"[ablation-ET] {len(splits)} outer folds, {n_trees} trees, no hyper-parameter search")
    results = []
    for name, blocks in chosen.items():
        use = [c for b in blocks for c in cols[b]]
        done = Path(outdir) / f"ablation_cv_{safe_name(name)}.csv"
        if skip_done and done.exists():
            print(f"[ablation-ET] {name}: found {done.name}, skipping"); results.append(pd.read_csv(done)); continue
        c2 = type(cfg)(**{**asdict(cfg), "mi_top_k": min(cfg.mi_top_k, len(use))})
        Xu = X[use]; t0 = time.time()
        out = Parallel(n_jobs=cfg.n_jobs)(delayed(_et_one_fold)(Xu, y, tr, va, c2, n_trees, cfg.random_seed)
                                           for _, _, _, tr, va, _ in splits)
        cv_df = pd.DataFrame([dict(split=sid, repeat=r, fold=f, model="extra_trees_fixed", roc_auc=a, pr_auc=b,
                                   n_features_used=k, config=name)
                              for (sid, r, f, _, _, _), (a, b, k) in zip(splits, out)])
        cv_df.to_csv(done, index=False); results.append(cv_df)
        print(f"[ablation-ET] {name}: {len(use)} features ({cv_df.n_features_used.mean():.0f} used), "
              f"ROC-AUC {cv_df.roc_auc.mean():.3f} +/- {cv_df.roc_auc.std():.3f} ({(time.time() - t0) / 60:.1f} min)")
    return pd.concat(results, ignore_index=True)

### Cell 15: redundancy functions

**What runs:** functions for CKA, effective dimensionality, ridge R², CCA, Spearman and the residualised-ESM test. `redundancy_report` is the same as before, except that it now also writes its results to `redundancy_results.json`. Definitions only.

In [ ]:
# ---------------------------------------------------------------- R3.4 ----------------------------
def _z(M):
    M = np.nan_to_num(np.asarray(M, float))
    M = M - M.mean(0); s = M.std(0); s[s == 0] = 1
    return M / s


def effective_dim(M):
    ev = PCA().fit(_z(M)).explained_variance_
    return int(np.searchsorted(np.cumsum(ev) / ev.sum(), 0.90) + 1), float(ev.sum() ** 2 / (ev ** 2).sum())


def linear_cka(A, B):
    A, B = _z(A), _z(B)
    return np.linalg.norm(A.T @ B, "fro") ** 2 / (np.linalg.norm(A.T @ A, "fro") * np.linalg.norm(B.T @ B, "fro"))


def cka_with_null(A, B, n_perm=200, seed=0):
    rng = np.random.default_rng(seed); obs = linear_cka(A, B)
    null = [linear_cka(A, np.asarray(B)[rng.permutation(len(B))]) for _ in range(n_perm)]
    return float(obs), float(np.mean(null)), float(np.percentile(null, 95))


def cv_ridge_r2(Xsrc, Ytgt, y, groups, n_splits=5, seed=42, alphas=(1, 10, 100, 1000, 1e4)):
    """Out-of-fold R2 of every column of Ytgt predicted from Xsrc (group-aware folds)."""
    Xs, Yt = np.asarray(Xsrc, float), np.asarray(Ytgt, float)
    oof = np.zeros_like(Yt)
    for tr, te in StratifiedGroupKFold(n_splits, shuffle=True, random_state=seed).split(Xs, y, groups):
        sx = StandardScaler().fit(Xs[tr]); sy = StandardScaler().fit(Yt[tr])
        Xtr, Xte = np.nan_to_num(sx.transform(Xs[tr])), np.nan_to_num(sx.transform(Xs[te]))
        Ytr = sy.transform(Yt[tr]); n = len(tr)
        idx = np.random.default_rng(seed).permutation(n); cut = int(.8 * n)
        best, ba = -np.inf, alphas[0]
        for a in alphas:
            m = Ridge(alpha=a).fit(Xtr[idx[:cut]], Ytr[idx[:cut]])
            r = 1 - ((Ytr[idx[cut:]] - m.predict(Xtr[idx[cut:]])) ** 2).sum() / (Ytr[idx[cut:]] ** 2).sum()
            if r > best: best, ba = r, a
        oof[te] = sy.inverse_transform(Ridge(alpha=ba).fit(Xtr, Ytr).predict(Xte))
    ss_res = ((Yt - oof) ** 2).sum(0); ss_tot = ((Yt - Yt.mean(0)) ** 2).sum(0)
    return 1 - ss_res / np.where(ss_tot == 0, np.nan, ss_tot)


def cv_cca(A, B, y, groups, n_comp=5, pcs=30, n_splits=5, seed=42, n_perm=100):
    A, B = _z(A), _z(B)
    def run(Bm):
        pa = np.zeros((len(A), n_comp)); pb = np.zeros((len(A), n_comp))
        for tr, te in StratifiedGroupKFold(n_splits, shuffle=True, random_state=seed).split(A, y, groups):
            ra = PCA(min(pcs, len(tr) - 1)).fit(A[tr]); rb = PCA(min(pcs, len(tr) - 1)).fit(Bm[tr])
            cca = CCA(n_components=n_comp, max_iter=1000).fit(ra.transform(A[tr]), rb.transform(Bm[tr]))
            pa[te], pb[te] = cca.transform(ra.transform(A[te]), rb.transform(Bm[te]))
        return np.array([np.corrcoef(pa[:, i], pb[:, i])[0, 1] for i in range(n_comp)])
    obs = run(B); rng = np.random.default_rng(seed)
    null = np.array([run(B[rng.permutation(len(B))]) for _ in range(n_perm)])
    return obs, null.mean(0), np.percentile(null, 95, axis=0)


def max_abs_spearman(E, H):
    re = _z(stats.rankdata(np.asarray(E, float), axis=0)); rh = _z(stats.rankdata(np.nan_to_num(np.asarray(H, float)), axis=0))
    return np.abs((re.T @ rh) / len(re)).max(1)


def residualized_esm_test(E, H, y, groups, n_splits=5, seed=42):
    """AUC of: hand only / ESM raw / ESM residual (part not predictable from hand features) / combinations.
    Residualisation and scaling are fit on the training fold only."""
    E, H = np.asarray(E, float), np.nan_to_num(np.asarray(H, float))
    names = ["hand only", "ESM raw only", "ESM residual only", "hand + ESM raw", "hand + ESM residual"]
    res = {n: [] for n in names}
    for tr, te in StratifiedGroupKFold(n_splits, shuffle=True, random_state=seed).split(E, y, groups):
        sh = StandardScaler().fit(H[tr]); se = StandardScaler().fit(E[tr])
        Htr, Hte = np.nan_to_num(sh.transform(H[tr])), np.nan_to_num(sh.transform(H[te]))
        Etr, Ete = se.transform(E[tr]), se.transform(E[te])
        rg = Ridge(alpha=1000).fit(Htr, Etr)
        Rtr, Rte = Etr - rg.predict(Htr), Ete - rg.predict(Hte)
        sets = {"hand only": (Htr, Hte), "ESM raw only": (Etr, Ete), "ESM residual only": (Rtr, Rte),
                "hand + ESM raw": (np.hstack([Htr, Etr]), np.hstack([Hte, Ete])),
                "hand + ESM residual": (np.hstack([Htr, Rtr]), np.hstack([Hte, Rte]))}
        for n, (a, b) in sets.items():
            clf = LogisticRegression(C=0.005, class_weight="balanced", max_iter=3000).fit(a, y[tr])
            res[n].append(roc_auc_score(y[te], clf.predict_proba(b)[:, 1]))
    return pd.DataFrame(res)

def redundancy_report(X, y, groups, outdir, n_perm_cka=200, n_perm_cca=100):
    cols = split_blocks(X); y = np.asarray(y); groups = np.asarray(groups)
    E, M, C = X[cols["ESM2"]].values, X[cols["MHC_ANCHOR"]].values, X[cols["CLASSICAL"]].values
    H = np.hstack([M, C]); L = []; S = {'effdim': {}, 'cka': {}, 'r2': {}, 'cca': {}, 'gains': {}}
    p = lambda s="": (print(s), L.append(s))
    p("R3.4 REDUNDANCY ANALYSIS"); p("=" * 60)
    p(f"Block sizes: ESM2={E.shape[1]}  MHC_ANCHOR={M.shape[1]}  CLASSICAL={C.shape[1]}  (ESM2 share of all columns: "
      f"{100 * E.shape[1] / X.shape[1]:.1f}%)")
    p("\n1. Effective dimensionality (PCs for 90% variance, participation ratio)")
    for b, c in cols.items():
        k90, pr = effective_dim(X[c].values); S['effdim'][b] = {'n_features': len(c), 'pcs90': k90, 'participation_ratio': pr}; p(f"   {b:11s} PCs90={k90:4d}  participation ratio={pr:7.1f}")
    p("\n2. Linear CKA (observed | permutation-null mean | null 95th percentile)")
    for nm, (A, B) in {"ESM2 vs CLASSICAL": (E, C), "ESM2 vs MHC_ANCHOR": (E, M), "CLASSICAL vs MHC_ANCHOR": (C, M)}.items():
        o, nm_, n95 = cka_with_null(A, B, n_perm_cka); S['cka'][nm] = {'observed': o, 'null_mean': nm_, 'null_p95': n95}; p(f"   {nm:25s} {o:.3f} | {nm_:.3f} | {n95:.3f}")
    p("\n3. Out-of-fold R^2 of each ESM2 dimension predicted from ... (group-aware 5-fold ridge)")
    r2 = {"hand-engineered (MHC+classical)": cv_ridge_r2(H, E, y, groups),
          "classical only": cv_ridge_r2(C, E, y, groups), "MHC_ANCHOR only": cv_ridge_r2(M, E, y, groups)}
    for nm, r in r2.items():
        S['r2'][nm] = {'mean': float(np.nanmean(r)), 'median': float(np.nanmedian(r)), 'share_gt_0.5': float(np.nanmean(r > .5)), 'share_lt_0.1': float(np.nanmean(r < .1))}
        p(f"   from {nm:32s} mean R2={np.nanmean(r):.3f}  median={np.nanmedian(r):.3f}  "
          f"share R2>0.5: {100 * np.nanmean(r > .5):.1f}%  share R2<0.1: {100 * np.nanmean(r < .1):.1f}%")
    p("\n4. Reverse direction: hand-engineered features predicted from ESM2")
    rb = cv_ridge_r2(E, H, y, groups)
    S['r2']['reverse: hand features from ESM2'] = {'mean': float(np.nanmean(rb)), 'median': float(np.nanmedian(rb)), 'share_gt_0.5': float(np.nanmean(rb > .5)), 'share_lt_0.1': float(np.nanmean(rb < .1))}
    p(f"   mean R2={np.nanmean(rb):.3f}  median={np.nanmedian(rb):.3f}  share R2>0.5: {100 * np.nanmean(rb > .5):.1f}%")
    p("\n5. Largest |Spearman| of each ESM2 dimension with any hand-engineered feature")
    m = max_abs_spearman(E, H)
    p(f"   median={np.median(m):.3f}  share>0.5: {100 * np.mean(m > .5):.1f}%  share>0.7: {100 * np.mean(m > .7):.1f}%")
    p("\n6. Held-out CCA between ESM2 and hand-engineered features (first 5 components)")
    obs, nmean, n95 = cv_cca(E, H, y, groups, n_perm=n_perm_cca)
    S['cca'] = {'observed': obs.tolist(), 'null_mean': nmean.tolist(), 'null_p95': n95.tolist()}
    for i in range(len(obs)): p(f"   comp {i + 1}: observed {obs[i]:.3f} | null mean {nmean[i]:.3f} | null 95th pct {n95[i]:.3f}")
    p("\n7. Label complementarity (ROC-AUC per outer fold, mean +/- SD; same logistic learner everywhere)")
    comp = residualized_esm_test(E, H, y, groups)
    for c in comp.columns: p(f"   {c:22s} {comp[c].mean():.3f} +/- {comp[c].std():.3f}")
    d = nb_paired(comp["hand + ESM residual"], comp["hand only"])
    S['gains']['hand + ESM residual vs hand only'] = dict(zip(['delta', 'ci_lo', 'ci_hi', 'p_corrected'], map(float, d)))
    p(f"   gain of (hand + ESM residual) over hand only: {d[0]:+.3f}  95% CI [{d[1]:+.3f}, {d[2]:+.3f}]  corrected p={d[3]:.3f}")
    d = nb_paired(comp["hand + ESM raw"], comp["hand only"])
    S['gains']['hand + ESM raw vs hand only'] = dict(zip(['delta', 'ci_lo', 'ci_hi', 'p_corrected'], map(float, d)))
    p(f"   gain of (hand + ESM raw)      over hand only: {d[0]:+.3f}  95% CI [{d[1]:+.3f}, {d[2]:+.3f}]  corrected p={d[3]:.3f}")
    (Path(outdir) / "redundancy_report.txt").write_text("\n".join(L))
    pd.DataFrame({"esm_dim": cols["ESM2"], "r2_from_hand": r2["hand-engineered (MHC+classical)"],
                  "r2_from_classical": r2["classical only"], "r2_from_mhc_anchor": r2["MHC_ANCHOR only"],
                  "max_abs_spearman_with_hand": m}).to_csv(Path(outdir) / "redundancy_esm_dim_r2.csv", index=False)
    comp.to_csv(Path(outdir) / "redundancy_label_complementarity.csv", index=False)
    S['block_sizes'] = {'ESM2': int(E.shape[1]), 'MHC_ANCHOR': int(M.shape[1]), 'CLASSICAL': int(C.shape[1]), 'total': int(X.shape[1])}
    (Path(outdir) / "redundancy_results.json").write_text(json.dumps(S, indent=1))
    p(f"\nSaved: {Path(outdir) / 'redundancy_report.txt'}")

### Cell 16: Data preparation functions

**What runs:** `prepare_data` (filtering, similarity groups, conflict resolution and building the feature matrix from the CSV) and the functions that save and load the feature matrix. Definitions only.

In [ ]:
import json
def prepare_data(cfg):
    """Mirrors steps 1-6 of run_full_pipeline() in your notebook (without any CV)."""
    df = pd.read_csv(cfg.input_csv)
    print(f"[load] {len(df)} rows from {cfg.input_csv}")
    df, _ = restrict_to_assay_supported(df, cfg)
    df, _ = restrict_to_tcell_assay(df, cfg)
    seqs = df[cfg.seq_col].astype(str).map(clean_sequence).tolist()
    groups = compute_similarity_groups(seqs, cfg.similarity_threshold)
    if cfg.resolve_conflicts:
        df, groups, _ = resolve_label_conflicts(df, groups, cfg)
    valid = df[cfg.seq_col].astype(str).map(clean_sequence).map(is_valid_peptide)
    if (~valid).any():
        df = df.loc[valid].reset_index(drop=True); groups = groups[valid.to_numpy()]
    feat_df = build_feature_matrix(df, cfg)
    drop_cols = [c for c in [cfg.id_col, cfg.label_col, cfg.seq_col] if c in feat_df.columns]
    X = feat_df.drop(columns=drop_cols)
    y = feat_df[cfg.label_col].to_numpy().astype(int)
    groups = np.asarray(groups)[:len(feat_df)]
    meta = pd.DataFrame({"META_dataset_id": df[cfg.id_col].values if cfg.id_col in df.columns else np.arange(len(df)),
                         "META_Peptide": df[cfg.seq_col].values,
                         "META_Organism": df["Organism"].values if "Organism" in df.columns else "",
                         "META_MHC_Class": df[cfg.mhc_class_col].values if cfg.mhc_class_col in df.columns else ""})
    return X, y, groups, meta


def save_features(X, y, groups, meta, path):
    fm = pd.concat([X.reset_index(drop=True), meta.reset_index(drop=True)], axis=1)
    fm["label"] = y; fm["group"] = groups
    fm.to_csv(path, index=False)
    print(f"[features] saved {fm.shape} to {path}")


def load_features(path):
    fm = pd.read_csv(path)
    X = fm[[c for c in fm.columns if c not in ("label", "group") and not c.startswith("META_")]]
    return X, fm["label"].to_numpy().astype(int), fm["group"].to_numpy(), fm

### Cell 17: Settings

**What runs:** only sets variables. Paths are already set for your PC. `N_JOBS` uses all logical CPU cores (in fast mode the 15 outer folds run in parallel, so more cores means faster); if RAM runs out, lower it. `LEARNER` chooses the fast fixed-ExtraTrees ablation or the slow tuned ensemble. `FAST = True` is a short test run only. To run only some configurations, put their names in `ABLATION_ONLY`, for example `["FULL", "MHC_ANCHOR + CLASSICAL", "MHC_ANCHOR only", "ESM2 only"]`; they run in that order.

In [ ]:
DATA_CSV      = r"H:\Peptide\review-1\ESM2\main_dataset_for_ml_model.csv"
OUT_DIR       = r"H:\Peptide\review-1\ESM2\r13_r34_out"     # feature_matrix_718.csv lives here (shared by all modes)
FAST          = False       # True: test run only, do not report these numbers in the paper
LEARNER       = "extra_trees_fixed"   # "extra_trees_fixed" = fast, no search | "tuned_ensemble" = slow, weighted ensemble
ET_TREES      = 500         # trees of the fixed Extra Trees learner
ESM_MODEL     = "esm2_t30_150M_UR50D"   # 640-dim, the model used in the manuscript
SEARCH_ITER   = 35          # reported run: 35 (used by the tuned mode)
REPEATS       = 3           # reported run: 3
PERM_CKA      = 200
PERM_CCA      = 100
N_JOBS        = os.cpu_count() or 4     # all logical cores
REUSE_FEATURES = True       # reuse an existing feature_matrix_718.csv, so ESM-2 is not recomputed
ABLATION_ONLY = None        # None = all 7, or e.g. ["FULL", "ESM2 only"]
OVERWRITE     = False       # True: discard earlier ablation results and recompute
if FAST:
    SEARCH_ITER, REPEATS, PERM_CKA, PERM_CCA = 10, 1, 20, 10
MODEL_NAME  = "extra_trees_fixed" if LEARNER == "extra_trees_fixed" else "weighted_ensemble"
MODEL_LABEL = "fixed Extra Trees, no search" if LEARNER == "extra_trees_fixed" else "tuned weighted ensemble"
print(dict(FAST=FAST, LEARNER=LEARNER, ESM_MODEL=ESM_MODEL, SEARCH_ITER=SEARCH_ITER, REPEATS=REPEATS, N_JOBS=N_JOBS))

### Cell 18: Output folders, data check, CPU thread settings

**What runs:** creates the output folder (named by mode: `_et` or `_tuned`, plus `_fast` for a test run, so different modes never mix), checks that the CSV or an existing `feature_matrix_718.csv` is there, and tells PyTorch to use all CPU threads. Prints whether a GPU is available.

In [ ]:
from pathlib import Path
suffix = ("_et" if LEARNER == "extra_trees_fixed" else "_tuned") + ("_fast" if FAST else "")
BASE = Path(OUT_DIR); BASE.mkdir(parents=True, exist_ok=True)          # shared feature matrix
OUT = Path(OUT_DIR + suffix); OUT.mkdir(parents=True, exist_ok=True)   # results of this run
FEAT_PATH = BASE / "feature_matrix_718.csv"
have_features = REUSE_FEATURES and FEAT_PATH.exists()
if not have_features and not Path(DATA_CSV).exists():
    raise FileNotFoundError(f"Data file not found: {DATA_CSV}")
import torch
torch.set_num_threads(N_JOBS)
print("Results folder:", OUT)
print("Feature matrix:", FEAT_PATH, "(found, will be reused)" if have_features else "(not found, will be built from the CSV)")
print("CPU threads:", N_JOBS, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (ESM-2 will run on CPU)")

### Cell 19: Build the feature matrix (ESM-2 runs here)

**What runs:** filters the data (T-cell assay, conflict resolution), computes similarity groups, then computes all features and the ESM-2 embeddings and saves `feature_matrix_718.csv`. The first time, the ESM-2 model is downloaded, which takes a while. On later runs the saved file is loaded instead. At the end the block sizes are printed. If `X = (718, 1373)` and ESM2 = 640 do not appear, a warning is shown. In that case check the ESM model name and the data file.

In [ ]:
cfg = Config(input_csv=DATA_CSV, outdir=str(OUT), n_jobs=N_JOBS,
             use_esm_embeddings=True, esm_model_name=ESM_MODEL, esm_device="", esm_batch_size=(32 if torch.cuda.is_available() else 16),
             random_search_iter=SEARCH_ITER, outer_repeats=REPEATS, outer_folds=5,
             require_tcell_assay=True, use_similarity_cv="auto", save_explainability_artifacts=False)

fpath = FEAT_PATH
if REUSE_FEATURES and fpath.exists():
    print("[features] loading", fpath)
    X, y, groups, _ = load_features(fpath)
else:
    X, y, groups, meta = prepare_data(cfg)
    save_features(X, y, groups, meta, fpath)

cols = split_blocks(X)
print("[check] X =", X.shape, "| blocks =", {b: len(c) for b, c in cols.items()},
      "| peptides =", len(y), "| positives =", int(y.sum()), "| similarity clusters =", len(np.unique(groups)))
if X.shape != (718, 1373) or len(cols["ESM2"]) != 640:
    print("WARNING: the manuscript had X = (718, 1373) and ESM2 = 640. Check ESM_MODEL and the data file.")

### Cell 20: R3.4: run the redundancy analysis

**What runs:** measures the collinearity between ESM-2 and the hand-engineered features: effective dimensionality, linear CKA (with a permutation null), out-of-fold ridge R² of every ESM dimension, the reverse-direction R², maximum Spearman correlation, held-out CCA and the residualised-ESM test. Output: `redundancy_report.txt`, `redundancy_esm_dim_r2.csv`, `redundancy_label_complementarity.csv` and `redundancy_results.json`. Moderately heavy because of the permutations and the CCA.

In [ ]:
t0 = time.time()
redundancy_report(X, y, groups, OUT, PERM_CKA, PERM_CCA)
print(f"\n[redundancy] done in {(time.time() - t0) / 60:.1f} min")

### Cell 21: run the ablation

**What runs:** the ablation for each of the 7 configurations, one after another. With `LEARNER = "extra_trees_fixed"` (default) every configuration runs one fixed Extra Trees model on the 15 outer folds, with the 15 folds computed in parallel. This is the fast mode. With `LEARNER = "tuned_ensemble"` it runs the full nested CV (hyper-parameter search, weighted ensemble), which is very slow. After each configuration finishes, `ablation_cv_<name>.csv` is saved (tuned mode also saves `ablation_oof_<name>.csv`). If you stop the kernel, rerun Cells 2 to 19 and then this cell. Configurations that are already finished are skipped. Keep the PC from sleeping while it runs.

In [ ]:
which = ABLATION_ONLY
if which:
    bad = [w for w in which if w not in ABLATION_CONFIGS]
    if bad: raise SystemExit(f"Unknown configuration(s) {bad}. Choose from {list(ABLATION_CONFIGS)}")
if FAST:
    print("NOTE: FAST run. Do not report these numbers in the paper.")
if LEARNER == "extra_trees_fixed":
    run_fixed_et_ablation(X, y, groups, cfg, OUT, which, skip_done=not OVERWRITE, n_trees=ET_TREES)
else:
    run_block_ablation(X, y, groups, cfg, OUT, which, skip_done=not OVERWRITE)

### Cell 22: Ablation summary tables

**What runs:** combines the saved results of all configurations into ROC-AUC and PR-AUC tables (with corrected CIs) and runs the paired tests (Nadeau-Bengio) for what each block adds. The row "adding ESM2 to MHC_ANCHOR + CLASSICAL" answers R2.3. A new row "MHC_ANCHOR only vs ESM2 only" is added, where delta = MHC minus ESM2. Output: `ablation_auc_table.csv`, `ablation_prauc_table.csv` and two `*_increments.csv` files.

In [ ]:
all_cv = load_ablation_results(OUT)
all_cv.to_csv(OUT / "ablation_cv_all_configs.csv", index=False)
print("Configurations found:", sorted(all_cv.config.unique()))
for metric, tag in (("roc_auc", "auc"), ("pr_auc", "prauc")):
    table, inc = summarize_ablation(all_cv, folds=cfg.outer_folds, model=MODEL_NAME, metric=metric)
    d = all_cv[all_cv.model == MODEL_NAME].pivot_table(index="split", columns="config", values=metric)
    if {"MHC_ANCHOR only", "ESM2 only"} <= set(d.columns):
        inc.loc["MHC_ANCHOR only vs ESM2 only"] = nb_paired(d["MHC_ANCHOR only"], d["ESM2 only"], cfg.outer_folds)
    inc = inc.rename(columns={0: "delta"}) if 0 in inc.columns else inc
    table.to_csv(OUT / f"ablation_{tag}_table.csv", index=False)
    inc.to_csv(OUT / f"ablation_{tag}_increments.csv")
    print(f"\n{metric}\n{table.round(3).to_string(index=False)}")
    print(f"\n{metric}: increments\n{inc.round(3).to_string()}")

### Cell 23: Figure 1: block ablation, ROC-AUC and PR-AUC

**What is drawn:** the mean AUC of each configuration with its corrected 95% CI, in two panels (ROC-AUC and PR-AUC). Red dots are configurations that contain ESM-2, blue dots are those without it, and black is FULL. It shows at a glance how much AUC moves when ESM-2 is added. File: `figures/fig1_ablation_auc.png/.pdf`.

In [ ]:
import json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from pathlib import Path

FIG = OUT / "figures"; FIG.mkdir(exist_ok=True)
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titlesize": 11, "axes.labelsize": 10, "figure.dpi": 110})
C_ESM, C_NO, C_FULL = "#c0504d", "#4a7ab5", "#2d2d2d"

def save_fig(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=300, bbox_inches="tight")
    fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight")
    print("Saved:", FIG / f"{name}.png")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), sharey=False)
for ax, (metric, tag, title) in zip(axes, (("roc_auc", "auc", "ROC-AUC"), ("pr_auc", "prauc", "PR-AUC"))):
    d = pd.read_csv(OUT / f"ablation_{tag}_table.csv").sort_values(f"{metric}_mean").reset_index(drop=True)
    m = d[f"{metric}_mean"].to_numpy()
    for i, r in d.iterrows():
        col = C_FULL if r.config == "FULL" else (C_ESM if "ESM2" in r.config else C_NO)
        ax.errorbar(m[i], i, xerr=[[m[i] - r.corrected_ci_lo], [r.corrected_ci_hi - m[i]]],
                    fmt="o", color=col, ecolor=col, capsize=3, ms=7, lw=1.4)
        ax.text(r.corrected_ci_hi + 0.004, i, f"{m[i]:.3f}", va="center", fontsize=9, color=col)
    ax.set_yticks(range(len(d))); ax.set_yticklabels(d.config)
    ax.set_xlabel(f"{title} (nested CV, corrected 95% CI)"); ax.set_title(title)
    ax.grid(axis="x", alpha=.25)
    ax.set_xlim(d.corrected_ci_lo.min() - 0.02, d.corrected_ci_hi.max() + 0.05)
fig.legend(handles=[Patch(color=C_ESM, label="with ESM-2"), Patch(color=C_NO, label="without ESM-2"),
                    Patch(color=C_FULL, label="FULL")], loc="lower center", ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.04))
fig.suptitle(f"Feature-block ablation ({MODEL_LABEL})", y=1.02)
fig.tight_layout(); save_fig(fig, "fig1_ablation_auc"); plt.show()

### Cell 24: Figure 2: incremental gain from ESM-2 and the other blocks

**What is drawn:** the AUC difference (delta) for each comparison with its corrected 95% CI, ROC-AUC and PR-AUC in separate panels. If the interval crosses the zero line, the difference is not meaningful. The row "adding ESM2 to MHC_ANCHOR + CLASSICAL" is the R2.3 question (full model with and without ESM-2), and "MHC_ANCHOR only vs ESM2 only" is the 0.818 vs 0.746 comparison. The p-values are in the labels. File: `figures/fig2_ablation_increments.png/.pdf`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, (tag, title) in zip(axes, (("auc", "ROC-AUC"), ("prauc", "PR-AUC"))):
    d = pd.read_csv(OUT / f"ablation_{tag}_increments.csv", index_col=0)
    d = d.iloc[::-1]
    for i, (lab, r) in enumerate(d.iterrows()):
        col = "#555555" if " vs " in lab else (C_ESM if "ESM2" in lab else C_NO)
        ax.errorbar(r.delta, i, xerr=[[r.delta - r.ci_lo], [r.ci_hi - r.delta]], fmt="s", color=col,
                    ecolor=col, capsize=3, ms=6, lw=1.4)
        ax.text(r.ci_hi + 0.004, i + 0.22, f"{r.delta:+.3f} (p={r.p_corrected:.3f})", fontsize=8.5, color=col)
    ax.axvline(0, color="grey", lw=1, ls="--")
    ax.set_yticks(range(len(d))); ax.set_yticklabels(d.index)
    ax.set_xlabel(f"Δ {title} (corrected 95% CI)"); ax.set_title(title)
    ax.grid(axis="x", alpha=.25)
    lo, hi = d.ci_lo.min(), d.ci_hi.max(); ax.set_xlim(lo - 0.02, hi + 0.13)
    ax.set_ylim(-0.6, len(d) + 0.2)
fig.suptitle("Incremental contribution of each feature block", y=1.02)
fig.tight_layout(); save_fig(fig, "fig2_ablation_increments"); plt.show()

### Cell 25: Figure 3: ESM-2 vs hand-engineered features, collinearity

**What is drawn:** four panels. (a) Linear CKA: the bar is the observed value and the black mark is the 95th percentile of the permutation null. (b) ECDF of how well each ESM-2 dimension can be predicted (out-of-fold R²) from the hand features, the classical features or the MHC features. (c) Held-out CCA, first 5 components, observed vs null. (d) Histogram of the maximum |Spearman| of each ESM dimension with any hand feature. File: `figures/fig3_redundancy_overview.png/.pdf`.

In [ ]:
S = json.loads((OUT / "redundancy_results.json").read_text())
r2d = pd.read_csv(OUT / "redundancy_esm_dim_r2.csv")
fig, ax = plt.subplots(2, 2, figsize=(11, 8))

# (a) CKA
a = ax[0, 0]; names = list(S["cka"].keys()); obs = [S["cka"][k]["observed"] for k in names]; n95 = [S["cka"][k]["null_p95"] for k in names]
a.bar(range(len(names)), obs, color="#7a9cc6", width=.55)
a.scatter(range(len(names)), n95, color="k", marker="_", s=900, lw=2, label="null 95th percentile", zorder=3)
for i, v in enumerate(obs): a.text(i, v + 0.01, f"{v:.2f}", ha="center", fontsize=9)
a.set_xticks(range(len(names))); a.set_xticklabels([n.replace(" vs ", "\nvs\n") for n in names], fontsize=9)
a.set_ylabel("linear CKA"); a.set_title("(a) Block similarity (CKA)"); a.legend(frameon=False, fontsize=8)

# (b) ECDF of R2
b = ax[0, 1]
for col, lab, c in (("r2_from_hand", "MHC + classical", "#c0504d"), ("r2_from_classical", "classical only", "#4a7ab5"),
                    ("r2_from_mhc_anchor", "MHC_ANCHOR only", "#6aa84f")):
    v = np.sort(r2d[col].dropna().to_numpy()); b.plot(np.clip(v, -0.5, 1), np.arange(1, len(v) + 1) / len(v), label=lab, color=c, lw=1.8)
b.axvline(0.5, color="grey", ls="--", lw=1)
b.set_xlabel("out-of-fold R² (ESM-2 dimension predicted from ...)"); b.set_ylabel("fraction of ESM-2 dimensions (≤ x)")
b.set_title("(b) ESM-2 dimensions predictable from hand features"); b.legend(frameon=False, fontsize=8)

# (c) CCA
c_ = ax[1, 0]; k = np.arange(1, len(S["cca"]["observed"]) + 1)
c_.bar(k - 0.18, S["cca"]["observed"], width=.36, color="#c0504d", label="observed (held-out)")
c_.bar(k + 0.18, S["cca"]["null_p95"], width=.36, color="#bbbbbb", label="null 95th percentile")
c_.set_xticks(k); c_.set_xlabel("canonical component"); c_.set_ylabel("held-out correlation")
c_.set_title("(c) Held-out CCA, ESM-2 vs hand features"); c_.set_ylim(top=1.3); c_.legend(frameon=False, fontsize=8, loc="upper right")

# (d) Spearman
d_ = ax[1, 1]; sp = r2d["max_abs_spearman_with_hand"].dropna()
d_.hist(sp, bins=30, color="#7a9cc6", edgecolor="white")
for x in (0.5, 0.7): d_.axvline(x, color="grey", ls="--", lw=1)
d_.set_xlabel("max |Spearman| (ESM-2 dimension vs any hand feature)"); d_.set_ylabel("number of ESM-2 dimensions")
d_.set_title(f"(d) median = {sp.median():.2f}")
fig.suptitle("Redundancy between ESM-2 embeddings and hand-engineered features", y=1.0)
fig.tight_layout(); save_fig(fig, "fig3_redundancy_overview"); plt.show()

### Cell 26: Figure 4: does ESM-2 carry new information about the label

**What is drawn:** per-fold ROC-AUC of five feature sets, all with the same logistic learner: hand only, ESM raw only, ESM residual only (what is left of ESM-2 after removing the part the hand features can explain), hand + ESM raw, and hand + ESM residual. Each dot is one fold and the black bar is the mean. The text under the plot gives the gain of each combination over hand only, with the corrected p-value. File: `figures/fig4_label_complementarity.png/.pdf`.

In [ ]:
comp = pd.read_csv(OUT / "redundancy_label_complementarity.csv")
S = json.loads((OUT / "redundancy_results.json").read_text())
order = ["hand only", "ESM raw only", "ESM residual only", "hand + ESM raw", "hand + ESM residual"]
fig, ax = plt.subplots(figsize=(8.5, 4.6))
rng = np.random.default_rng(0)
for i, nme in enumerate(order):
    v = comp[nme].to_numpy()
    ax.scatter(i + rng.uniform(-.12, .12, len(v)), v, color="#7a9cc6", s=28, alpha=.85, zorder=2)
    ax.hlines(v.mean(), i - .28, i + .28, color="k", lw=2.2, zorder=3)
    ax.text(i, v.max() + 0.008, f"{v.mean():.3f}", ha="center", fontsize=9)
ax.set_xticks(range(len(order))); ax.set_xticklabels([o.replace(" + ", "\n+ ").replace(" only", "\nonly") for o in order], fontsize=9)
ax.set_ylabel("ROC-AUC (outer fold)"); ax.grid(axis="y", alpha=.25)
g1, g2 = S["gains"]["hand + ESM residual vs hand only"], S["gains"]["hand + ESM raw vs hand only"]
ax.set_title("Label information carried by ESM-2 beyond the hand-engineered features")
fig.text(0.5, -0.04, f"hand + ESM residual − hand only: {g1['delta']:+.3f} [{g1['ci_lo']:+.3f}, {g1['ci_hi']:+.3f}], p = {g1['p_corrected']:.3f}   |   "
         f"hand + ESM raw − hand only: {g2['delta']:+.3f} [{g2['ci_lo']:+.3f}, {g2['ci_hi']:+.3f}], p = {g2['p_corrected']:.3f}",
         ha="center", fontsize=8.5)
fig.tight_layout(); save_fig(fig, "fig4_label_complementarity"); plt.show()

### Cell 27: Zip all results

**What runs:** puts the results folder (tables, figures, per-configuration CSVs and the redundancy files) into one zip file next to it. For the response letter, `redundancy_report.txt` and the `ablation_*` files are enough.

In [ ]:
import shutil
zip_path = shutil.make_archive(str(Path(OUT_DIR).parent / ("r13_r34_results_" + OUT.name)), "zip", str(OUT))
print("Zip created:", zip_path)